# Part 1

# Step 0: Problem Setup and Initial Point Generation

In [1]:
import numpy as np

# Problem data
mu = np.array([0.1073, 0.0737, 0.0627])       # Expected returns
Sigma = np.array([                           # Covariance matrix
    [0.02778, 0.00387, 0.00021],
    [0.00387, 0.01112, -0.00020],
    [0.00021, -0.00020, 0.00115]
])
delta = 4.0  # Chosen risk aversion parameter

Q = 4 * Sigma
c = mu.copy()                # As we're minimizing, this is used directly
A = np.ones((1, 3))
b = np.array([1.0])
n = len(mu)

# Initial Point Generation (Slide 40 with corrected deltas)
def initial_point(Q, c, A, b):
    e = np.ones(n)

    # Compute intermediate values
    At = A.T
    inv_AAt = np.linalg.inv(A @ At)
    pi_bar = inv_AAt @ (A @ c)
    z_bar = c - At @ pi_bar
    x_bar = At @ inv_AAt @ b

    # Basic deltas
    delta_x = max(-1.5 * np.min(x_bar), 0)
    delta_z = max(-1.5 * np.min(z_bar), 0)

    x_tilde = x_bar + delta_x * e
    z_tilde = z_bar + delta_z * e

    # Scalar products for bar-deltas
    numerator = np.dot(x_tilde, z_tilde)
    delta_bar_x = delta_x + numerator / (2 * np.sum(z_tilde))
    delta_bar_z = delta_z + numerator / (2 * np.sum(x_tilde))

    # Final initial points
    x0 = x_bar + delta_bar_x * e
    #x0 = x_bar + delta_bar_x
    z0 = z_bar + delta_bar_z * e
    #z0 = z_bar + delta_bar_z
    pi0 = pi_bar

    return x0, pi0, z0


# Step 1

Solve Predictor Step (affine direction)

In [2]:
def predictor_step_qp(Q, A, b, c, x, pi, z):
    n = len(x)
    m = A.shape[0]
    e = np.ones(n)

    # Residuals
    r_d = Q @ x + A.T @ pi + z - c
    r_p = A @ x - b
    r_c = x * z  # Complementarity

    # Diagonal matrices
    X = np.diag(x)
    Z = np.diag(z)

    # Build KKT matrix (NOTE the -Q)
    KKT = np.block([
        [Q, A.T, np.eye(n)],
        [A, np.zeros((m, m)), np.zeros((m, n))],
        [Z, np.zeros((n, m)), X]
    ])

    rhs = -np.concatenate([r_d, r_p, r_c])

    # Solve system
    sol = np.linalg.solve(KKT, rhs)

    dx_aff = sol[:n]
    dpi_aff = sol[n:n + m]
    dz_aff = sol[n + m:]

    return dx_aff, dpi_aff, dz_aff


# Step 2

Compute step lengths and centering parameter τ

In [3]:
def step_length_and_tau_single(x, z, dx_aff, dz_aff):
    n = len(x)

    # Min step to stay positive in x and z
    alpha_aff_x = min(
        [-x[i] / dx_aff[i] for i in range(n) if dx_aff[i] < 0],
        default=1.0
    )
    alpha_aff_z = min(
        [-z[i] / dz_aff[i] for i in range(n) if dz_aff[i] < 0],
        default=1.0
    )

    alpha_aff = min(1.0, alpha_aff_x, alpha_aff_z)

    # Current duality gap
    y = np.dot(x, z) / n

    # Affine step duality gap
    x_aff = x + alpha_aff * dx_aff
    z_aff = z + alpha_aff * dz_aff
    y_aff = np.dot(x_aff, z_aff) / n

    # Centering parameter
    tau = (y_aff / y) ** 3

    return alpha_aff, tau, y, y_aff


Solve for corrector step

In [4]:
def corrector_step_qp(Q, A, b, c, x, pi, z, dx_aff, dz_aff, tau, y):
    n = len(x)
    m = A.shape[0]
    e = np.ones(n)

    # Residuals
    r_d = Q @ x + A.T @ pi + z - c
    r_p = A @ x - b

    # Diagonal matrices
    X = np.diag(x)
    Z = np.diag(z)
    Dx = np.diag(dx_aff)
    Dz = np.diag(dz_aff)

    DxDz_e = Dx @ Dz @ e
    rc = x * z + DxDz_e - tau * y * e  # add two D back

    rhs = -np.concatenate([r_d, r_p, rc])

    KKT = np.block([
        [Q, A.T, np.eye(n)],
        [A, np.zeros((m, m)), np.zeros((m, n))],
        [Z, np.zeros((n, m)), X]
    ])

    sol = np.linalg.solve(KKT, rhs)

    dx_corr = sol[:n]
    dpi_corr = sol[n:n + m]
    dz_corr = sol[n + m:]

    return dx_corr, dpi_corr, dz_corr


# Step 3

computing the final step size with dampening (η ∈ [0.9, 1]) and updating variables

In [5]:
def step_and_update(x, pi, z, dx, dpi, dz, eta=0.95):
    n = len(x)

    # Compute individual maximum steps
    alpha_x_max = min([-x[i] / dx[i] for i in range(n) if dx[i] < 0], default=1.0)
    alpha_z_max = min([-z[i] / dz[i] for i in range(n) if dz[i] < 0], default=1.0)

    # Multiply η inside min as per the step length formulas
    alpha = min(1.0, eta * alpha_x_max, eta * alpha_z_max)

    # Update variables
    x_new = x + alpha * dx
    pi_new = pi + alpha * dpi
    z_new = z + alpha * dz

    return x_new, pi_new, z_new, alpha


looping using def functions above

In [6]:
def solve_qp_primal_dual(Q, c, A, b, tol=1e-8, max_iter=1000, eta=0.95, verbose=True):
    """
    Solves the convex quadratic program:
        minimize   2xᵗQx - cᵗx  => set risk aversion = 4
        subject to Ax = b, x ≥ 0

    using the Predictor-Corrector Primal-Dual Interior Point Method (Slides 31–40).

    Parameters:
        Q, c, A, b: QP problem data
        tol: convergence tolerance
        max_iter: maximum number of iterations
        eta: dampening factor for step length (e.g., 0.95)
        verbose: whether to print iteration log

    Returns:
        x, pi, z: optimal primal and dual variables
        history: list of iterates (x, pi, z, primal_residual, dual_residual, complementarity, alpha)
    """

    # Step 0: Generate initial feasible interior point (x > 0, z > 0, Ax = b, ATπ + z = c)
    x0, pi0, z0 = initial_point(Q, c, A, b)
    x = x0
    pi = pi0
    z = z0

    print("initial point: ", x)

    history = []

    # Main iteration loop
    for k in range(max_iter):
        # Step 1: Predictor step (compute affine directions dx_aff, dpi_aff, dz_aff)
        dx_aff, dpi_aff, dz_aff = predictor_step_qp(Q, A, b, c, x, pi, z)

        # Step 2: Compute single affine step length and centering parameter tau
        alpha_aff, tau, y, y_aff = step_length_and_tau_single(x, z, dx_aff, dz_aff)

        # Step 2: Corrector step (with second-order correction using DxDz term)
        dx, dpi, dz = corrector_step_qp(Q, A, b, c, x, pi, z, dx_aff, dz_aff, tau, y)

        # Step 3: Compute final step length (dampened) and update variables
        x, pi, z, alpha = step_and_update(x, pi, z, dx, dpi, dz, eta)

        # Step 3: Compute stopping criteria quantities
        primal_res = np.linalg.norm(A @ x - b)                      # ||Ax - b||
        dual_res = np.linalg.norm(Q @ x + A.T @ pi + z - c)            # ||Qx + Aᵗπ + z - c||
        comp = np.dot(x, z)                                        # xᵗz (complementarity)

        # Save iteration history for later analysis
        history.append((x.copy(), pi.copy(), z.copy(), primal_res, dual_res, comp, alpha))

        # Optionally print iteration log
        if verbose:
            print(f"Iter {k}:")
            print(f"  ||Ax - b|| = {primal_res:.2e}")
            print(f"  ||Qx + Aᵗπ + z - c|| = {dual_res:.2e}")
            print(f"  xᵗz = {comp:.2e}")
            print(f"  α = {alpha:.4f}")
            print(f"  x = {x}")
            print(f"  π = {pi}")
            print(f"  z = {z}")

        # Check stopping condition: all residuals and complementarity are small
        if primal_res < tol and dual_res < tol and comp < tol:
            break

    return x, pi, z, history


Executing defined functions to find optimal result

In [7]:
x_star, pi_star, z_star, iterates = solve_qp_primal_dual(Q, c, A, b)

initial point:  [0.5 0.5 0.5]
Iter 0:
  ||Ax - b|| = 4.79e-02
  ||Qx + Aᵗπ + z - c|| = 1.29e-02
  xᵗz = 8.41e-03
  α = 0.9043
  x = [0.41401349 0.48572953 0.14813146]
  π = [0.05092098]
  z = [0.01282417 0.00170833 0.0153533 ]
Iter 1:
  ||Ax - b|| = 3.87e-03
  ||Qx + Aᵗπ + z - c|| = 1.04e-03
  xᵗz = 2.62e-03
  α = 0.9193
  x = [0.46358347 0.51125945 0.02902211]
  π = [0.04427171]
  z = [4.39112508e-03 8.54166667e-05 1.86546660e-02]
Iter 2:
  ||Ax - b|| = 2.16e-04
  ||Qx + Aᵗπ + z - c|| = 5.81e-05
  xᵗz = 1.86e-04
  α = 0.9442
  x = [0.49957826 0.49917705 0.00146042]
  π = [0.04379079]
  z = [3.13076084e-04 4.27083333e-06 1.89011853e-02]
Iter 3:
  ||Ax - b|| = 1.08e-05
  ||Qx + Aᵗπ + z - c|| = 2.91e-06
  xᵗz = 9.36e-06
  α = 0.9500
  x = [5.02112875e-01 4.97824896e-01 7.30211415e-05]
  π = [0.04378542]
  z = [1.56773208e-05 2.13552876e-07 1.88916747e-02]
Iter 4:
  ||Ax - b|| = 5.40e-07
  ||Qx + Aᵗπ + z - c|| = 1.45e-07
  xᵗz = 4.68e-07
  α = 0.9500
  x = [5.02239790e-01 4.97757098e-01 3

In [8]:
# Compute and print the true (maximization) objective from the original problem
print("Iter | Expected Return | Variance       | Objective Value")
print("----------------------------------------------------------")

for i, (x, _, _, _, _, _, _) in enumerate(iterates):
    expected_return = mu @ x
    variance = x @ Sigma @ x
    objective = expected_return - 0.5 * delta * variance  # original maximization form
    print(f"{i+1:4d} | {expected_return:16.8f} | {variance:13.8f} | {objective:16.8f}")



Iter | Expected Return | Variance       | Objective Value
----------------------------------------------------------
   1 |       0.08950976 |    0.00896398 |       0.07158179
   2 |       0.08924201 |    0.01071196 |       0.06781810
   3 |       0.09048566 |    0.01163435 |       0.06721697
   4 |       0.09057098 |    0.01169441 |       0.06718217
   5 |       0.09057526 |    0.01169742 |       0.06718041
   6 |       0.09057547 |    0.01169758 |       0.06718032
   7 |       0.09057548 |    0.01169758 |       0.06718031


Compare with function solution

In [9]:
import cvxpy as cp
import numpy as np

def solve_with_cvxpy(Q, c, A, b):
    n = len(c)
    x = cp.Variable(n)

    objective = cp.Minimize((2) * cp.quad_form(x, Q) - c @ x)
    constraints = [A @ x == b, x >= 0]
    problem = cp.Problem(objective, constraints)
    problem.solve()

    return x.value, problem.value

x_cvxpy, obj_cvxpy = solve_with_cvxpy(Q, c, A, b)
print("x from cvxpy (quadprog equivalent):", x_cvxpy)
print("Objective value (max form):", mu @ x_cvxpy - 2 * x_cvxpy @ Sigma @ x_cvxpy)

x from cvxpy (quadprog equivalent): [0.11013737 0.11726218 0.77260045]
Objective value (max form): 0.06635042554986054


reason of the difference: Q is positive definite but with one eigenvalue approximately equal to 0. This makes the objective surface is very flat in one direction, i.e., close to being degenerate. Because the surface is so flat, small numerical differences or different search paths cause different optima.

In [10]:
np.linalg.eigvalsh(Q)

array([0.0045708 , 0.04108504, 0.11454416])

# Part 2

In [11]:
def initial_point(Q, c, A, b):
    number = c.shape[0]
    e = np.ones(number)
    A = A.reshape(1, -1)
    At = A.T
    AAt = A @ At

    pi_bar = np.linalg.solve(AAt, A @ c)
    z_bar = c - At @ pi_bar
    x_bar = At @ np.linalg.solve(AAt, b)

    delta_x = max(-1.5 * np.min(x_bar), 0)
    delta_z = max(-1.5 * np.min(z_bar), 0)

    epsilon = 1e-4  # ensure strict positivity
    x_tilde = np.maximum(x_bar + delta_x * e, epsilon)
    z_tilde = np.maximum(z_bar + delta_z * e, epsilon)

    numerator = np.dot(x_tilde, z_tilde)
    denom_z = np.sum(z_tilde)
    denom_x = np.sum(x_tilde)

    delta_bar_x = delta_x + numerator / (2 * denom_z)
    delta_bar_z = delta_z + numerator / (2 * denom_x)

    x0 = x_bar + delta_bar_x * e
    z0 = z_bar + delta_bar_z * e
    pi0 = pi_bar

    return x0, pi0, z0


Generate random covariance matrix

In [12]:
def generate_qp_instance(n, seed=None):
    if seed is not None:
        np.random.seed(seed)

    mu = np.ones(n)  # fixed
    A = np.random.randn(n, n)
    Sigma = A + A.T + n * np.eye(n)  # mimic MATLAB code to ensure positive definite

    Q = 4 * Sigma
    c = mu.copy()
    A_eq = np.ones((1, n))
    b_eq = np.array([1.0])

    return Q, c, A_eq, b_eq, mu, Sigma


test generated covariance matrix

In [13]:
def compare_solvers(n, seed=0):
    Q, c, A, b, mu, Sigma = generate_qp_instance(n, seed)

    # Solve with your IPM
    x_ipm, pi_ipm, z_ipm, iters_ipm = solve_qp_primal_dual(Q, c, A, b)

    # Solve with cvxpy
    x_cvx, obj_cvx = solve_with_cvxpy(Q, c, A, b)

    # Compute objective from IPM
    expected_return = mu @ x_ipm
    variance = x_ipm @ Sigma @ x_ipm
    obj_ipm = expected_return - 0.5 * delta * variance  # original maximization form

    print(f"n = {n}")
    print(f"IPM Objective:    {obj_ipm:.6f}")
    #print(f"CVXPY Objective:  {obj_cvx:.6f}")
    print("CVXPY Objective: ", mu @ x_cvx - 2 * x_cvx @ Sigma @ x_cvx)

    print("x from IPM:", x_ipm)
    print("x from cvxpy (quadprog equivalent):", x_cvx)


Run for different size covariance matrix

In [14]:
compare_solvers(5)

initial point:  [0.3 0.3 0.3 0.3 0.3]
Iter 0:
  ||Ax - b|| = 2.19e-02
  ||Qx + Aᵗπ + z - c|| = 1.15e+00
  xᵗz = 8.43e-06
  α = 0.9563
  x = [0.14001818 0.16079844 0.20197586 0.3012203  0.21785229]
  π = [-6.24173816]
  z = [1.73682663e-05 1.37563620e-05 8.08133642e-06 2.50000000e-06
 6.41950930e-06]
Iter 1:
  ||Ax - b|| = 1.09e-03
  ||Qx + Aᵗπ + z - c|| = 5.74e-02
  xᵗz = 4.18e-07
  α = 0.9501
  x = [0.13306803 0.15475104 0.19771691 0.3012727  0.21428313]
  π = [-6.55636005]
  z = [9.12658013e-07 7.07694733e-07 4.07560648e-07 1.25000000e-07
 3.22592711e-07]
Iter 2:
  ||Ax - b|| = 5.46e-05
  ||Qx + Aᵗπ + z - c|| = 2.87e-03
  xᵗz = 2.09e-08
  α = 0.9500
  x = [0.13272101 0.15444909 0.19750425 0.30127531 0.21410492]
  π = [-6.57206921]
  z = [4.56394268e-08 3.53875673e-08 2.03785263e-08 6.25000000e-09
 1.61298688e-08]
Iter 3:
  ||Ax - b|| = 2.73e-06
  ||Qx + Aᵗπ + z - c|| = 1.44e-04
  xᵗz = 1.04e-09
  α = 0.9500
  x = [0.13270366 0.154434   0.19749362 0.30127544 0.21409601]
  π = [-6.5728

In [15]:
compare_solvers(10)

initial point:  [0.15 0.15 0.15 0.15 0.15 0.15 0.15 0.15 0.15 0.15]
Iter 0:
  ||Ax - b|| = 0.00e+00
  ||Qx + Aᵗπ + z - c|| = 4.13e-15
  xᵗz = 8.25e-06
  α = 1.0000
  x = [0.05888369 0.01836846 0.10992232 0.20869654 0.06896917 0.12819199
 0.17364762 0.09441707 0.12196957 0.01693357]
  π = [-2.5925602]
  z = [2.00272055e-05 4.00820781e-05 5.14725776e-06 9.23493167e-06
 1.61689965e-05 2.63471383e-06 2.82071497e-06 8.44333399e-06
 3.32389975e-06 4.09260963e-05]
Iter 1:
  ||Ax - b|| = 0.00e+00
  ||Qx + Aᵗπ + z - c|| = 1.38e-15
  xᵗz = 4.12e-07
  α = 0.9500
  x = [0.05888386 0.01836918 0.10992199 0.20869599 0.06896952 0.12819175
 0.17364726 0.09441683 0.1219692  0.01693444]
  π = [-2.59255237]
  z = [1.00136028e-06 2.00410405e-06 2.57362888e-07 4.61746584e-07
 8.08449831e-07 1.31735692e-07 1.41035749e-07 4.22166700e-07
 1.66194988e-07 2.04630507e-06]
Iter 2:
  ||Ax - b|| = 2.22e-16
  ||Qx + Aᵗπ + z - c|| = 1.59e-15
  xᵗz = 2.06e-08
  α = 0.9500
  x = [0.05888387 0.01836921 0.10992197 0.20869

In [16]:
compare_solvers(20)

initial point:  [0.075 0.075 0.075 0.075 0.075 0.075 0.075 0.075 0.075 0.075 0.075 0.075
 0.075 0.075 0.075 0.075 0.075 0.075 0.075 0.075]
Iter 0:
  ||Ax - b|| = 1.84e-02
  ||Qx + Aᵗπ + z - c|| = 9.72e-01
  xᵗz = 8.43e-06
  α = 0.9632
  x = [0.0375045  0.04907336 0.07464445 0.06671501 0.02887501 0.032747
 0.05768376 0.03802717 0.04626614 0.04104925 0.03768807 0.06463532
 0.04040403 0.0820316  0.0527515  0.04567338 0.05501115 0.07399346
 0.05453266 0.0391146 ]
  π = [-2.3157024]
  z = [1.54738062e-05 8.70234629e-06 2.50000000e-06 3.13227712e-06
 2.21334051e-05 1.89752861e-05 5.26606607e-06 1.51145957e-05
 1.01184672e-05 1.31364968e-05 1.53470691e-05 3.49023050e-06
 1.35446663e-05 2.95521507e-06 7.06703532e-06 1.04360855e-05
 6.18622288e-06 2.50819722e-06 6.36487001e-06 1.43833978e-05]
Iter 1:
  ||Ax - b|| = 9.20e-04
  ||Qx + Aᵗπ + z - c|| = 4.86e-02
  xᵗz = 4.18e-07
  α = 0.9501
  x = [0.03614195 0.04813114 0.07463147 0.06641382 0.02719886 0.03121159
 0.05705441 0.03668362 0.0452219  0.

In [17]:
compare_solvers(50)

initial point:  [0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03
 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03
 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03
 0.03 0.03 0.03 0.03 0.03 0.03 0.03 0.03]
Iter 0:
  ||Ax - b|| = 1.80e-02
  ||Qx + Aᵗπ + z - c|| = 1.51e+00
  xᵗz = 7.95e-06
  α = 0.9640
  x = [0.02195113 0.02188776 0.00999623 0.01879048 0.02360463 0.01868048
 0.02813385 0.03165377 0.0176238  0.02145853 0.02459672 0.02246069
 0.0247281  0.0286139  0.02381528 0.02207014 0.01064132 0.02291039
 0.01907546 0.01291887 0.01742445 0.02746343 0.01734681 0.0195183
 0.01729538 0.01851726 0.02118961 0.01284928 0.02138255 0.01427403
 0.02383208 0.01616676 0.02260018 0.01823957 0.02698637 0.02640221
 0.02564319 0.01920303 0.01384454 0.01842185 0.02187938 0.01585315
 0.01715269 0.02383226 0.02076512 0.01897231 0.01759744 0.02287857
 0.01860261 0.01626298]
  π = [-2.51822587]
  z = [6.12281104e-06 6.18183416e-06 2.54504295e

In [18]:
compare_solvers(100)

initial point:  [0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015 0.015
 0.015 0.015 0.015 0.015]
Iter 0:
  ||Ax - b|| = 1.34e-02
  ||Qx + Aᵗπ + z - c|| = 1.57e+00
  xᵗz = 7.33e-06
  α = 0.9732
  x = [0.00960638 0.01037473 0.010369   0.01178269 0.01064171 0.01049769
 0.01196297 0.01324941 0.00912889 0.00725299 0.00987798 0.006567
 0.01072773 0.01066624 0.00981095 0.01113586 0.00993568 0.01239516
 0.01167668 0.00895594 0.00767807 0.01145542 0.01074533 0.01032688
 0.010

In [19]:
compare_solvers(1000)

initial point:  [0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015
 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.0015 0.00

In [20]:
compare_solvers(2000)

initial point:  [0.00075 0.00075 0.00075 ... 0.00075 0.00075 0.00075]
Iter 0:
  ||Ax - b|| = 2.22e-16
  ||Qx + Aᵗπ + z - c|| = 3.60e-13
  xᵗz = 6.10e-06
  α = 1.0000
  x = [0.00052354 0.00053548 0.00052092 ... 0.00050948 0.000473   0.00048448]
  π = [-2.99979588]
  z = [5.10374800e-06 4.63587817e-06 5.20993791e-06 ... 5.68753203e-06
 7.36567649e-06 6.81215025e-06]
Iter 1:
  ||Ax - b|| = 0.00e+00
  ||Qx + Aᵗπ + z - c|| = 1.95e-13
  xᵗz = 3.05e-07
  α = 0.9500
  x = [0.00052354 0.00053548 0.00052092 ... 0.00050948 0.000473   0.00048448]
  π = [-2.99979008]
  z = [2.55187400e-07 2.31793909e-07 2.60496895e-07 ... 2.84376601e-07
 3.68283825e-07 3.40607512e-07]
Iter 2:
  ||Ax - b|| = 1.11e-16
  ||Qx + Aᵗπ + z - c|| = 1.93e-13
  xᵗz = 1.53e-08
  α = 0.9500
  x = [0.00052354 0.00053548 0.00052092 ... 0.00050948 0.000473   0.00048448]
  π = [-2.99978979]
  z = [1.27593700e-08 1.15896954e-08 1.30248448e-08 ... 1.42188301e-08
 1.84141912e-08 1.70303756e-08]
Iter 3:
  ||Ax - b|| = 0.00e+00
  ||Qx 